# 03 · Análise exploratória

**Objetivo:** entender os dados coletados antes de modelar. A pergunta principal: **o quanto a posição de largada já explica o resultado da corrida?** Esse vai ser o "baseline" que o modelo precisa superar.

## Setup

In [1]:
import pandas as pd 
import glob

## 1. Carregando os dados
Leio todos os CSVs salvos pelo notebook 02 e junto numa tabela só. Quando novas temporadas forem coletadas, basta rodar de novo.

In [5]:
arquivos = glob.glob("../data/dados_*.csv")
dados = pd.concat([pd.read_csv(arq) for arq in arquivos], ignore_index=True)

print(arquivos)
dados.shape


['../data\\dados_2018.csv', '../data\\dados_2019.csv', '../data\\dados_2020.csv', '../data\\dados_2021.csv', '../data\\dados_2022.csv', '../data\\dados_2023.csv', '../data\\dados_2024.csv', '../data\\dados_2026.csv']


(3353, 25)

## 2. Qualidade dos dados
Antes de analisar, verifico onde há valores vazios e se eles têm explicação. Vazios esperados (como Q2 e Q3 de quem foi eliminado) são mantidos; vazios sem explicação indicam falha na coleta.

In [6]:
# Quantidade de valores vazios por coluna
dados.isna().sum()

DriverNumber             0
BroadcastName         2475
Abbreviation             0
DriverId                20
TeamName                 0
TeamColor             2475
TeamId                  20
FirstName                0
LastName                 0
FullName                 0
HeadshotUrl           2502
CountryCode           2914
Position                22
ClassifiedPosition      20
GridPosition            22
Time                  1196
Status                  20
Points                  20
Laps                    20
QualiPosition           28
Q1                      65
Q2                     903
Q3                    1745
Ano                      0
GP                       0
dtype: int64

## 3. Limpeza das colunas
Removo as colunas que não ajudam a prever o resultado: as que vêm quase sempre vazias (a fonte não fornece) e as que só repetem a identificação do piloto.

In [10]:
colunas_removidas = [
    "BroadcastName", "TeamColor", "HeadshotUrl", "CountryCode", "FirstName", "LastName", "FullName", "DriverNumber"
]

dados = dados.drop(columns=colunas_removidas)
dados.columns

Index(['Abbreviation', 'DriverId', 'TeamName', 'TeamId', 'Position',
       'ClassifiedPosition', 'GridPosition', 'Time', 'Status', 'Points',
       'Laps', 'QualiPosition', 'Q1', 'Q2', 'Q3', 'Ano', 'GP'],
      dtype='object')

### Dicionário de colunas

**Como funciona a classificação (Q1, Q2 e Q3):** no sábado, os pilotos disputam a classificação, que define a ordem de largada de domingo. Ela funciona como um mata-mata em três partes: no **Q1** todos correm e os mais lentos são eliminados; no **Q2** mais alguns são eliminados; no **Q3** os 10 mais rápidos disputam a pole position. Em cada parte, conta a melhor volta do piloto. Quem foi eliminado não tem tempo nas partes seguintes, por isso há valores vazios.

| Coluna | Significado | Quando é conhecida |
|---|---|---|
| `Abbreviation` | Sigla do piloto (ex.: NOR, VER) | Antes da corrida |
| `DriverId` | Identificador único do piloto (chave) | Antes da corrida |
| `TeamName` / `TeamId` | Equipe do piloto | Antes da corrida |
| `GridPosition` | Posição de largada | Antes da corrida |
| `QualiPosition` | Posição na classificação de sábado | Antes da corrida |
| `Q1`, `Q2`, `Q3` | Melhor volta em cada parte da classificação | Antes da corrida |
| `Position` | **Posição final na corrida (o que queremos prever)** | Depois da corrida |
| `ClassifiedPosition` | Posição oficial (`R` para quem abandonou) | Depois da corrida |
| `Status` | Como terminou (`Finished`, `Retired`...) | Depois da corrida |
| `Points` | Pontos ganhos | Depois da corrida |
| `Laps` | Voltas completadas | Depois da corrida |
| `Time` | Tempo de prova (ou diferença para o vencedor) | Depois da corrida |
| `Ano` / `GP` | Temporada e Grande Prêmio | Antes da corrida |

> Colunas conhecidas **depois** da corrida não podem ser usadas para prever o resultado: seria vazamento de dados.

## 4. Correções nos dados

### Largadas do pit lane
Nesta fonte, quem larga do pit lane aparece com `GridPosition = 0`. Como 0 sugeriria "à frente da pole", troco pelo último lugar do grid daquela corrida e guardo a informação numa coluna própria (`LargouPitLane`).

In [11]:
# Quantas largadas do pit lane (GridPosition = 0)?
dados[dados["GridPosition"] == 0][["Ano", "GP", "Abbreviation", "GridPosition", "Position"]]

,Ano,GP,Abbreviation,GridPosition,Position
199,2018,British Grand Prix,HAR,0.0,20.0
469,2019,Chinese Grand Prix,ALB,0.0,10.0
489,2019,Azerbaijan Grand Prix,RAI,0.0,10.0
495,2019,Azerbaijan Grand Prix,KUB,0.0,16.0
496,2019,Azerbaijan Grand Prix,GAS,0.0,17.0
512,2019,Spanish Grand Prix,HUL,0.0,13.0
556,2019,Canadian Grand Prix,MAG,0.0,17.0
597,2019,Austrian Grand Prix,RUS,0.0,18.0
676,2019,Belgian Grand Prix,KUB,0.0,17.0
694,2019,Italian Grand Prix,RAI,0.0,15.0


In [12]:
# Marca quem largou do pit lane e troca o 0 pelo último lugar do grid daquela corrida
dados["LargouPitLane"] = (dados["GridPosition"] == 0).astype(int)

n_carros = dados.groupby(["Ano", "GP"])["DriverId"].transform("count")
dados["GridPosition"] = dados["GridPosition"].where(dados["GridPosition"] != 0, n_carros)

(dados["GridPosition"] == 0).sum()

np.int64(0)

### Linhagem das equipes
Algumas equipes mudaram de nome ao longo dos anos (troca de dono ou patrocinador), mas são a mesma estrutura. Crio a coluna `Equipe`, que agrupa os nomes antigos sob o nome atual, para que o histórico de cada equipe não seja quebrado.

In [19]:
# Agrupa os nomes antigos de cada equipe sob o nome atual
linhagem = {
    "force_india": "aston_martin",
    "racing_point": "aston_martin",
    "toro_rosso": "rb",
    "alphatauri": "rb",
    "sauber": "audi",
    "alfa": "audi",
    "renault": "alpine",
}

dados["Equipe"] = dados["TeamId"].replace(linhagem)
dados.groupby("Equipe")["Ano"].unique()

Equipe
alpine          [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2026]
aston_martin    [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2026]
audi            [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2026]
cadillac                                                  [2026]
ferrari         [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2026]
haas            [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2026]
mclaren         [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2026]
mercedes        [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2026]
rb              [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2026]
red_bull        [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2026]
williams        [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2026]
Name: Ano, dtype: object